# Inspect a parsed document

A helper for looking at what `ai_parse_document` produced for a PDF and how it came out as Markdown. It is **not** part of the job. Open it in the workspace and run it for one document at a time.

What it shows:

1. **Overview**: pages, elements by type, and any parser error.
2. **Page view**: the real PDF page with a coloured box around every parsed element (its type and reading order), like the Databricks parse viewer.
3. **Elements table**: the page's elements in reading order, with type, confidence and content.
4. **Parsed vs Markdown**: the text the parser found for the page next to the Markdown that `silver_pdfs` holds for it.
5. **Marker check**: whether the document's `<!-- page N -->` markers match its pages (duplicates, gaps).
6. **All documents** (optional): the marker check for every converted document, to find the ones to look at.

**How the boxes are drawn.** The parser returns each element's box in pixels of the page rendered at **200 DPI** (a US-letter page is 1700 x 2200). The notebook renders the page from the PDF in `bronze_pdfs` at 200 DPI and draws the boxes on it.

**Page numbers.** `page` is counted 1, 2, 3 over the pages that have content, the same numbering the silver notebook uses for `<!-- page N -->`. `page_id` is the parser's own 0-based id, shown next to it.

In [ ]:
%pip install -q pymupdf
dbutils.library.restartPython()

In [ ]:
import base64, html, io, re
import pandas as pd

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
dbutils.widgets.text("doc", "BOEING_2022_10K")   # part of the PDF file name
dbutils.widgets.text("page", "1")                # page to look at (1, 2, 3, ...)
dbutils.widgets.dropdown("run_all_docs", "no", ["no", "yes"])

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
doc = dbutils.widgets.get("doc")
page = int(dbutils.widgets.get("page"))
run_all_docs = dbutils.widgets.get("run_all_docs") == "yes"

bronze = f"{catalog}.{schema}.bronze_pdfs"
silver = f"{catalog}.{schema}.silver_pdfs"
DPI = 200  # the parser's coordinates are in pixels of the page rendered at 200 DPI

## 1. Load the document and get an overview

Finds the document by a part of its file name (the `doc` widget). If more than one file matches, use a longer name.

In [ ]:
rows = spark.table(silver).filter(f"path LIKE '%{doc}%'").select("path", "markdown", "num_parts").collect()
assert len(rows) == 1, f"{len(rows)} documents match '{doc}': {[r.path for r in rows]}. Use a more specific name."
path, markdown, num_parts = rows[0].path, rows[0].markdown, rows[0].num_parts
print(path)

# One row per parsed element, in reading order.
els = spark.sql(f"""
SELECT e.pos AS pos,
       coalesce(e.value:bbox[0].page_id::INT, 0) AS page_id,
       e.value:type::STRING AS type,
       e.value:confidence::DOUBLE AS confidence,
       variant_get(e.value, '$.bbox[0].coord', 'ARRAY<INT>') AS coord,
       e.value:content::STRING AS content,
       e.value:description::STRING AS description
FROM (SELECT parsed FROM {silver} WHERE path = '{path}') p,
     LATERAL variant_explode(p.parsed:document:elements) e
ORDER BY e.pos
""").toPandas()

error = spark.sql(f"SELECT CAST(parsed:error_status AS STRING) FROM {silver} WHERE path = '{path}'").first()[0]
page_ids = sorted(int(v) for v in els["page_id"].unique())  # plain ints: the PDF library rejects numpy ints
els["page"] = els["page_id"].map({pid: i + 1 for i, pid in enumerate(page_ids)})

print(f"elements: {len(els)}   pages with content: {len(page_ids)}   parts: {num_parts}   parser errors: {error}")
display(els.groupby("type").agg(elements=("pos", "count"), avg_chars=("content", lambda s: round(s.fillna("").str.len().mean()))).sort_values("elements", ascending=False).reset_index())

## 2. The page, with the parsed elements drawn on it

Renders the PDF page (from the bytes stored in `bronze_pdfs`) and draws a box around each element, coloured by type and numbered in reading order. Use it to check that titles, tables, headers and footers were found where they should be, and that nothing on the page was missed (text with no box).

In [ ]:
import fitz
from PIL import Image, ImageDraw, ImageFont

COLORS = {
    "title": "#d62728", "section_header": "#ff7f0e", "text": "#1f77b4", "table": "#2ca02c",
    "figure": "#9467bd", "caption": "#8c564b", "page_header": "#7f7f7f", "page_footer": "#7f7f7f",
    "page_number": "#bcbd22", "footnote": "#17becf", "list_item": "#e377c2",
}
DEFAULT_COLOR = "#000000"

pdf_bytes = spark.table(bronze).filter(f"path = '{path}'").select("content").first()[0]
pdf = fitz.open(stream=bytes(pdf_bytes), filetype="pdf")
print(f"PDF has {len(pdf)} pages; the parser found content on {len(page_ids)} of them")

def render_page(page_no, dpi=DPI, show_order=True):
    """PDF page `page_no` (1, 2, 3 over pages with content) as a PIL image with the element boxes drawn on it."""
    page_id = page_ids[page_no - 1]
    pix = pdf[page_id].get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    draw = ImageDraw.Draw(img)
    for _, e in els[els["page"] == page_no].iterrows():
        if e["coord"] is None or len(e["coord"]) != 4:
            continue
        color = COLORS.get(e["type"], DEFAULT_COLOR)
        x1, y1, x2, y2 = [int(v) for v in e["coord"]]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=4)
        label = f"{e['pos']} {e['type']}" if show_order else e["type"]
        draw.rectangle([x1, max(y1 - 26, 0), x1 + 12 * len(label) + 8, max(y1, 26)], fill=color)
        draw.text((x1 + 4, max(y1 - 24, 0)), label, fill="white")
    return img

def show_image(img, width=900):
    img = img.copy(); img.thumbnail((width, width * 2))
    buf = io.BytesIO(); img.save(buf, format="PNG")
    displayHTML(f'<img src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}" style="max-width:100%">')

legend = " ".join(f'<span style="background:{c};color:white;padding:2px 6px;margin:2px;border-radius:3px">{t}</span>' for t, c in COLORS.items())
displayHTML(legend)
show_image(render_page(page))

## 3. The page's elements

The elements of the chosen page in reading order. `pos` is the position in the whole document, `coord` is `[x1, y1, x2, y2]` in pixels at 200 DPI. Tables are given as HTML in `content`, and a figure may have a `description` generated by the parser.

In [ ]:
page_els = els[els["page"] == page].copy()
page_els["content"] = page_els["content"].fillna("").str.slice(0, 200)
display(page_els[["pos", "type", "confidence", "coord", "content", "description"]])

## 4. Parsed text next to the Markdown

Left: the text the parser found on the page, element by element. Right: the Markdown that `silver_pdfs` holds for the same page (everything between this page's `<!-- page N -->` marker and the next one). Look for dropped text, reworded text, a wrong heading level or a table that lost rows.

In [ ]:
markers = [(m.start(), m.end(), int(m.group(1))) for m in re.finditer(r"<!-- page ([0-9]+) -->", markdown or "")]

def markdown_of_page(n):
    """All text after a `<!-- page n -->` marker up to the next marker of a different page (duplicates are joined)."""
    parts = []
    for i, (s, e, num) in enumerate(markers):
        if num == n:
            end = markers[i + 1][0] if i + 1 < len(markers) else len(markdown)
            parts.append(markdown[e:end].strip().removesuffix("---").strip())
    return "\n\n[... same page marker again ...]\n\n".join(parts) if parts else "(no marker for this page)"

parsed_text = "\n\n".join(f"[{r.type}] {r.content}" for r in els[els["page"] == page].itertuples() if r.content)
col = "width:49%;display:inline-block;vertical-align:top;white-space:pre-wrap;font-family:monospace;font-size:12px;padding:6px;border:1px solid #888;"
displayHTML(
    f'<div style="{col}"><b>Parsed (page {page})</b>\n\n{html.escape(parsed_text)}</div>'
    f'<div style="{col}"><b>Markdown (page {page})</b>\n\n{html.escape(markdown_of_page(page))}</div>'
)

## 5. Do the page markers match the pages?

The silver notebook asks the model to write one `<!-- page N -->` marker per page. This lists what is wrong with this document: pages that have no marker, markers that appear more than once, and markers outside the page range. After the marker check was added to the silver notebook, a freshly converted document should have none of these.

In [ ]:
found = [n for _, _, n in markers]
expected = list(range(1, len(page_ids) + 1))
missing = [n for n in expected if n not in found]
dups = sorted({n for n in found if found.count(n) > 1})
outside = sorted({n for n in found if n not in expected})
in_order = found == sorted(found)
print(f"pages with content: {len(expected)}   markers: {len(found)}")
print(f"missing markers : {missing or 'none'}")
print(f"repeated markers: {dups or 'none'}")
print(f"outside range   : {outside or 'none'}")
print(f"in page order   : {in_order}")
print("OK" if not (missing or dups or outside) and in_order and len(found) == len(expected) else "MARKERS DO NOT MATCH THE PAGES")

## 6. All documents (optional)

Set the `run_all_docs` widget to `yes` and rerun this cell. It compares, for every converted document, the number of markers with the number of pages that have content, and the size of the Markdown with the size of the parsed text. Sort by `marker_diff` or `ratio` to find the documents worth opening above.

- `marker_diff` is markers minus pages: positive means repeated markers, negative means missing ones. It should be 0.
- `ratio` is Markdown characters divided by parsed text characters. Around 0.95 is normal (HTML tables become shorter Markdown tables). Far below that, for example under 0.7, can mean dropped text.

In [ ]:
if run_all_docs:
    all_docs = spark.sql(f"""
    WITH e AS (
      SELECT path, sum(length(v.value:content::STRING)) AS content_chars,
             count(DISTINCT coalesce(v.value:bbox[0].page_id::INT, 0)) AS pages
      FROM {silver}, LATERAL variant_explode(parsed:document:elements) v GROUP BY path
    )
    SELECT regexp_extract(path, '[^/]+$', 0) AS doc, e.pages,
           size(regexp_extract_all(markdown, '<!-- page ([0-9]+) -->', 1)) AS markers,
           size(regexp_extract_all(markdown, '<!-- page ([0-9]+) -->', 1)) - e.pages AS marker_diff,
           round(length(markdown) / e.content_chars, 2) AS ratio
    FROM {silver} s JOIN e USING (path)
    WHERE markdown IS NOT NULL
    ORDER BY abs(marker_diff) DESC
    """)
    display(all_docs)
    print("documents with a marker mismatch:", all_docs.filter("marker_diff <> 0").count(), "of", all_docs.count())
else:
    print("Set the run_all_docs widget to yes to run this.")